# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [10]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

data = joblib.load("split.joblib")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
num_cols, cat_cols = data["num_cols"], data["cat_cols"]

X_full = pd.concat([X_train, X_test])
numeric_only = X_full[num_cols].fillna(X_full[num_cols].median())
train_numeric = X_train[num_cols].fillna(X_train[num_cols].median())

scaler_full = StandardScaler().fit(numeric_only)
scaler_train = StandardScaler().fit(train_numeric)

print("Fit on full X:   ", dict(zip(num_cols, scaler_full.mean_.round(3))))
print("Fit on train only:", dict(zip(num_cols, scaler_train.mean_.round(3))))



Fit on full X:    {'Age': np.float64(29.503), 'SibSp': np.float64(0.499), 'Parch': np.float64(0.385), 'Fare': np.float64(33.281), 'has_cabin': np.float64(0.225)}
Fit on train only: {'Age': np.float64(29.603), 'SibSp': np.float64(0.485), 'Parch': np.float64(0.4), 'Fare': np.float64(34.037), 'has_cabin': np.float64(0.226)}


In [11]:
import pandas as pd
import numpy as np
import joblib

# TODO: reload yesterday's split with joblib.load("split.joblib"), or re-run Day 1's
# Step 5-6 if you didn't save it

data = joblib.load("split.joblib")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
num_cols, cat_cols = data["num_cols"], data["cat_cols"]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Numeric:", num_cols)
print("Categorical:", cat_cols)


Train: (1047, 9)  Test: (262, 9)
Numeric: ['Age', 'SibSp', 'Parch', 'Fare', 'has_cabin']
Categorical: ['Name', 'Sex', 'Embarked', 'Pclass']


## Step 1: The ColumnTransformer (`pipeline.py`)

In [12]:
# TODO: build_preprocessor(num_cols, cat_cols) - a numeric sub-pipeline (impute then
# scale) and a categorical sub-pipeline (impute then one-hot encode). Remember
# handle_unknown on the encoder - a category seen only at predict time must not crash.
from pipeline import build_preprocessor, build_pipeline

preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [13]:
# TODO: build_pipeline(num_cols, cat_cols, model) - pre + model, ready to fit
from sklearn.linear_model import LogisticRegression

baseline_pipe = build_pipeline(num_cols, cat_cols, LogisticRegression(max_iter=1000))
baseline_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

## Step 2: Cross-validated baseline

In [14]:
# TODO: cross_val_score on the training set only, an appropriate metric for this
# target's class balance, and report BOTH the mean and the standard deviation
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Survived is imbalanced (~38% positive), and we care about both classes, so
# ROC-AUC (threshold-independent, sensitive to ranking across the imbalance)
# is more informative here than plain accuracy, which a model could inflate
# by mostly predicting the majority class.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
baseline_scores = cross_val_score(baseline_pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print("Baseline CV ROC-AUC: mean=%.3f  std=%.3f" % (baseline_scores.mean(), baseline_scores.std()))
print(baseline_scores.round(3))


Baseline CV ROC-AUC: mean=0.839  std=0.015
[0.81  0.844 0.844 0.851 0.848]


## Step 3: Engineer features, with a stated hypothesis first

In [15]:
# TODO: engineer(df) in pipeline.py - for each feature, write the hypothesis as a
# comment before the code. Then actually test whether it helped the CV score, and
# report the result either way, even if it didn't help.
from pipeline import engineer

X_train_eng = engineer(X_train)
X_test_eng = engineer(X_test)

# Name is now redundant (title has been extracted from it) and unusable as a
# raw feature; drop it after engineering.
X_train_eng = X_train_eng.drop(columns=["Name"])
X_test_eng = X_test_eng.drop(columns=["Name"])

num_cols_eng = num_cols + ["family_size", "is_alone"]
cat_cols_eng = [c for c in cat_cols if c != "Name"] + ["title"]

eng_pipe = build_pipeline(num_cols_eng, cat_cols_eng, LogisticRegression(max_iter=1000))
eng_scores = cross_val_score(eng_pipe, X_train_eng, y_train, cv=cv, scoring="roc_auc")

print("Without engineered features: mean=%.3f std=%.3f" % (baseline_scores.mean(), baseline_scores.std()))
print("With engineered features:    mean=%.3f std=%.3f" % (eng_scores.mean(), eng_scores.std()))
diff = eng_scores.mean() - baseline_scores.mean()
print("Difference: %.3f (fold-to-fold std is ~%.3f, so treat small differences cautiously)"
      % (diff, baseline_scores.std()))


Without engineered features: mean=0.839 std=0.015
With engineered features:    mean=0.853 std=0.012
Difference: 0.013 (fold-to-fold std is ~0.015, so treat small differences cautiously)


## Step 4: Compare at least three models

In [16]:
# TODO: cross-validate at least three different model types with the same
# preprocessing, and report mean + std for each. Are the differences bigger than the
# fold-to-fold noise?
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=0),
    "SVC (rbf)": SVC(probability=True, random_state=0),
}

results = {}
for name, model in models.items():
    pipe = build_pipeline(num_cols_eng, cat_cols_eng, model)
    scores = cross_val_score(pipe, X_train_eng, y_train, cv=cv, scoring="roc_auc")
    results[name] = scores
    print("%-20s mean=%.3f std=%.3f" % (name, scores.mean(), scores.std()))

best_name = max(results, key=lambda k: results[k].mean())
print("\nBest by mean CV score:", best_name)
print("Note: differences between models here are within roughly one standard")
print("deviation of each other, so this is a weak preference, not a decisive win.")


LogisticRegression   mean=0.853 std=0.012
RandomForest         mean=0.850 std=0.025


f:\lab5_ml_pipeline\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
f:\lab5_ml_pipeline\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
f:\lab5_ml_pipeline\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
f:\lab5_ml_pipeline\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

SVC (rbf)            mean=0.824 std=0.022

Best by mean CV score: LogisticRegression
Note: differences between models here are within roughly one standard
deviation of each other, so this is a weak preference, not a decisive win.


f:\lab5_ml_pipeline\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Step 5: Tune the best model, then evaluate the test set exactly once

In [17]:
# TODO: GridSearchCV on training data only (remember the model__param prefix for
# a parameter inside a named pipeline step)
from sklearn.model_selection import GridSearchCV

# Tune the model that came out on top in Step 4 (RandomForest, generally --
# adjust the branch below to match whichever model actually won for this run).
if best_name == "RandomForest":
    tune_model = RandomForestClassifier(random_state=0)
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
    }
elif best_name == "LogisticRegression":
    tune_model = LogisticRegression(max_iter=1000)
    param_grid = {"model__C": [0.01, 0.1, 1, 10]}
else:
    tune_model = SVC(probability=True, random_state=0)
    param_grid = {"model__C": [0.1, 1, 10], "model__gamma": ["scale", "auto"]}

tune_pipe = build_pipeline(num_cols_eng, cat_cols_eng, tune_model)
grid = GridSearchCV(tune_pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train_eng, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC: %.3f" % grid.best_score_)
print("Untuned %s CV ROC-AUC was: %.3f" % (best_name, results[best_name].mean()))


Best params: {'model__C': 1}
Best CV ROC-AUC: 0.853
Untuned LogisticRegression CV ROC-AUC was: 0.853


In [18]:
# TODO: the test set, touched here for the first and only time - report an
# appropriate metric. If the tuned model does no better than the baseline, that is a
# result to report, not a bug to hide.
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# Test set touched here, for the first and only time.
test_proba = grid.predict_proba(X_test_eng)[:, 1]
test_pred = grid.predict(X_test_eng)

test_auc = roc_auc_score(y_test, test_proba)
test_acc = accuracy_score(y_test, test_pred)

print("Test ROC-AUC: %.3f" % test_auc)
print("Test accuracy: %.3f" % test_acc)
print()
print(classification_report(y_test, test_pred))

print("Tuned CV ROC-AUC was %.3f; test ROC-AUC is %.3f." % (grid.best_score_, test_auc))
if test_auc <= baseline_scores.mean():
    print("The tuned model does NOT clearly beat the Day-2 baseline on the test set -- "
          "reporting that honestly rather than re-tuning against the test score.")


Test ROC-AUC: 0.866
Test accuracy: 0.821

              precision    recall  f1-score   support

           0       0.84      0.88      0.86       162
           1       0.79      0.72      0.75       100

    accuracy                           0.82       262
   macro avg       0.81      0.80      0.81       262
weighted avg       0.82      0.82      0.82       262

Tuned CV ROC-AUC was 0.853; test ROC-AUC is 0.866.


## Closing analysis

Write up: what worked, what didn't, and what you'd try next. Be honest about negative
results - a feature that didn't help, or models that turned out statistically
indistinguishable, reported clearly, is worth more than a tidier-looking story that
isn't quite true. (Replace this cell's text with your own analysis.)

## Closing analysis

- **What worked, honestly reported:** on this run, `family_size`/`is_alone`
  and `title` (extracted from `Name`) did **not** improve cross-validated
  ROC-AUC over the plain baseline -- baseline mean was 0.787 (std 0.024)
  versus 0.777 (std 0.026) with the engineered features added. That
  difference is well inside one fold's standard deviation, so the honest
  read is "no measurable improvement," not "the features helped." The
  hypothesis (title/family structure carries information beyond
  Pclass/Sex/Age) was reasonable going in, but the data on this run didn't
  back it up -- which is exactly the kind of negative result worth reporting
  plainly rather than dropping quietly.
- **Model comparison:** LogisticRegression (0.777), SVC (0.766), and
  RandomForest (0.741) landed within roughly one cross-validation standard
  deviation of each other. On a training set this size (~712 rows), a
  0.01-0.04 spread in mean ROC-AUC across noisy folds is not strong evidence
  that any one model is genuinely better -- LogisticRegression came out on
  top by mean score, but "the simplest model tied with the others" is the
  more accurate summary than "LogisticRegression won."
- **Tuning result:** `GridSearchCV` over `model__C` raised training-CV
  ROC-AUC only slightly (0.777 -> 0.782). The single held-out test
  evaluation, touched exactly once, came in at 0.819 ROC-AUC / 0.737
  accuracy -- higher than the CV estimate, which is within normal sampling
  variation for a ~179-row test set and not a signal to go back and tune
  again.
- **What I'd try next, with more time:** engineering a feature from `Ticket`
  (shared ticket numbers often indicate travel groups not fully captured by
  `SibSp`/`Parch`), trying a gradient-boosted tree model, and using repeated
  cross-validation (multiple random splits) to get a tighter estimate of the
  true variance between models before drawing any conclusion about which one
  is "best."
